In [5]:
# TẮT TOÀN BỘ MỌI CẢNH BÁO Ở CẤP ĐỘ HỆ THỐNG
import warnings
import os
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("=== BÀI 3: KHAI THÁC LUẬT KẾT HỢP (D2 & D3) ===\n")

try:
    # --- 1. XỬ LÝ D2 (HOTEL BOOKING) ---
    print("[1] Đang xử lý bộ dữ liệu D2 (Hotel Bookings)...")
    df2 = pd.read_csv('hotel_bookings.csv', nrows=5000)

    df2['lead_time_bin'] = pd.qcut(df2['lead_time'], q=3, labels=['Short', 'Medium', 'Long'])
    d2_basket = pd.get_dummies(df2[['hotel', 'meal', 'customer_type', 'lead_time_bin']]).astype(bool)

    freq2_c1 = apriori(d2_basket, min_support=0.1, use_colnames=True)
    rules2_c1 = association_rules(freq2_c1, metric="confidence", min_threshold=0.5)

    rules2_final = rules2_c1[rules2_c1['lift'] > 1.0].sort_values('lift', ascending=False).reset_index(drop=True)

    print(f"-> D2: Tìm thấy {len(rules2_final)} luật có giá trị (Lift > 1.0).")
    print("TOP 3 LUẬT MẠNH NHẤT CỦA D2:")
    print(rules2_final[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3))


    # --- 2. XỬ LÝ D3 (GROCERIES) ---
    print("\n" + "="*50)
    print("\n[2] Đang xử lý bộ dữ liệu D3 (Groceries)...")

    df3_raw = pd.read_csv('Groceries.csv')

    if 'Member_number' in df3_raw.columns and 'itemDescription' in df3_raw.columns:
        print("-> Nhận diện dữ liệu dạng Bảng. Đang gom nhóm thành Giỏ hàng...")
        transactions = df3_raw.groupby('Member_number')['itemDescription'].apply(list).tolist()
    else:
        print("-> Nhận diện dữ liệu dạng Giỏ thô.")
        with open('data/raw/Groceries.csv', 'r') as f:
            transactions = [line.strip().split(',') for line in f.readlines()]

    te = TransactionEncoder()
    d3_basket = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

    freq3_c1 = apriori(d3_basket, min_support=0.01, use_colnames=True)
    rules3_c1 = association_rules(freq3_c1, metric="confidence", min_threshold=0.05)

    rules3_final = rules3_c1[rules3_c1['lift'] > 1.0].sort_values('lift', ascending=False).reset_index(drop=True)

    print(f"-> D3: Tìm thấy {len(rules3_final)} luật có giá trị (Lift > 1.0).")
    print("TOP 3 LUẬT MẠNH NHẤT CỦA D3:")
    if len(rules3_final) > 0:
        print(rules3_final[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(3))
    else:
        print("Vẫn không tìm thấy luật. Cấu trúc dữ liệu có thể khác biệt.")

except FileNotFoundError as e:
    print(f"LỖI: Không tìm thấy file dữ liệu. Hãy kiểm tra lại thư mục data/raw/.\nChi tiết: {e}")
except Exception as e:
    print(f"LỖI HỆ THỐNG: {e}")

=== BÀI 3: KHAI THÁC LUẬT KẾT HỢP (D2 & D3) ===

[1] Đang xử lý bộ dữ liệu D2 (Hotel Bookings)...
-> D2: Tìm thấy 33 luật có giá trị (Lift > 1.0).
TOP 3 LUẬT MẠNH NHẤT CỦA D2:
                                 antecedents  \
0  (hotel_Resort Hotel, lead_time_bin_Short)   
1                      (lead_time_bin_Short)   
2                      (lead_time_bin_Short)   

                                         consequents  support  confidence  \
0                 (customer_type_Transient, meal_BB)   0.2372    0.698469   
1                 (customer_type_Transient, meal_BB)   0.2372    0.698469   
2  (customer_type_Transient, hotel_Resort Hotel, ...   0.2372    0.698469   

       lift  
0  1.259864  
1  1.259864  
2  1.259864  


[2] Đang xử lý bộ dữ liệu D3 (Groceries)...
-> Nhận diện dữ liệu dạng Bảng. Đang gom nhóm thành Giỏ hàng...
-> D3: Tìm thấy 13046 luật có giá trị (Lift > 1.0).
TOP 3 LUẬT MẠNH NHẤT CỦA D3:
                               antecedents  \
0                     (rolls/